# SimVerse — Train a Robot Arm on Colab GPU

This notebook trains a Panda robot arm to pick up a red cube from a desk.
After training, download the model and run it locally with the 3D viewer.

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Install SimVerse

In [ ]:
#@title Setup (click ▶ and wait ~2 min) { display-mode: "form" }
#@markdown Enter your GitHub repo URL:
REPO_URL = "https://github.com/amirkaplan/simulation-universe.git"  #@param {type:"string"}

import os, sys, subprocess

# 1. System dependencies for headless MuJoCo rendering
print("Installing system dependencies...")
subprocess.run(["apt-get", "install", "-y", "-qq", "libosmesa6-dev", "libgl1-mesa-glx"],
               capture_output=True, check=True)
os.environ["MUJOCO_GL"] = "osmesa"

# 2. Python dependencies
print("Installing Python packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "mujoco", "gymnasium", "stable-baselines3", "pydantic",
                "pyyaml", "shimmy", "mediapy", "setuptools", "wheel"],
               capture_output=True, check=True)

# 3. Clone or update the repo
REPO_DIR = "/content/simulation-universe"
if os.path.exists(REPO_DIR):
    print("Updating repo...")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, capture_output=True)
else:
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

# 4. Install simverse package
print("Installing simverse...")
result = subprocess.run([sys.executable, "-m", "pip", "install", ".[training]"],
                        capture_output=True, text=True)
if result.returncode != 0:
    print("pip install failed, using PYTHONPATH fallback...")
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# 5. Verify
import simverse
print(f"\nSimVerse {simverse.__version__} ready! GPU: {os.environ.get('MUJOCO_GL', 'default')}")

import torch
print(f"PyTorch device: {'cuda (' + torch.cuda.get_device_name() + ')' if torch.cuda.is_available() else 'cpu'}")

## 2. Verify — Preview the Environment

In [ ]:
import gymnasium as gym
import simverse.envs
import mediapy
import numpy as np

env = gym.make("SimVerse/DeskPickup-v0", render_mode="rgb_array")
obs, info = env.reset(seed=42)

# Render from 3 camera angles
frames = []
for cam in ["overhead", "side", "front"]:
    engine = env.unwrapped.engine
    frame = engine.render(width=480, height=360, camera_name=cam)
    frames.append(frame.rgb)

mediapy.show_images(frames, titles=["Overhead", "Side", "Front"], height=240)
env.close()
print(f"Observation keys: {list(obs.keys())}")
print(f"Action space: {env.action_space.shape} (7 joints + 1 gripper)")

## 3. Train the Policy

Adjust `total_timesteps` based on how long you want to wait:
- `100_000` → ~3 min (learns to reach toward the cube)
- `500_000` → ~15 min (learns to reach and maybe grasp)
- `1_000_000` → ~30 min (best chance at full pick-up)

In [ ]:
from simverse.training import Trainer, TrainingConfig, Algorithm

config = TrainingConfig(
    env_id="SimVerse/DeskPickup-v0",
    algorithm=Algorithm.PPO,
    total_timesteps=500_000,
    learning_rate=3e-4,
    device="cuda",
    seed=42,
)

trainer = Trainer(config)
trainer.setup()
model_path = trainer.train()
trainer.cleanup()

print(f"\nModel saved to: {model_path}")

## 4. Watch the Trained Policy (Video)

In [ ]:
from stable_baselines3 import PPO

trained_model = PPO.load(str(model_path))
env = gym.make("SimVerse/DeskPickup-v0", render_mode="rgb_array")

frames = []
obs, info = env.reset(seed=42)
total_reward = 0
successes = 0
episodes = 0

for step in range(1000):
    action, _ = trained_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    if step % 2 == 0:  # capture every other frame for smaller video
        frame = env.render()
        if frame is not None:
            frames.append(frame)

    if terminated or truncated:
        episodes += 1
        if info.get("success", False):
            successes += 1
        obs, info = env.reset()

env.close()

print(f"Episodes: {episodes}, Successes: {successes}, Total reward: {total_reward:.1f}")
mediapy.show_video(frames, fps=15, title="Trained Policy")

## 5. Evaluate Performance

In [ ]:
from simverse.training.evaluation import evaluate_policy

env = gym.make("SimVerse/DeskPickup-v0")
result = evaluate_policy(trained_model, env, n_episodes=20)
env.close()

print(f"Mean reward:  {result.mean_reward:.2f} ± {result.std_reward:.2f}")
print(f"Mean length:  {result.mean_length:.0f} steps")
print(f"Success rate: {result.success_rate:.1%}")

## 6. Download the Trained Model

Download and run locally with the interactive 3D viewer:
```bash
cd simulation-universe
python scripts/watch_trained.py --model path/to/final_model.zip
```

In [ ]:
import shutil
from google.colab import files

# The model is saved as a .zip file by SB3
model_zip = str(model_path) + ".zip"
if not os.path.exists(model_zip):
    # SB3 may have already added .zip extension
    model_zip = str(model_path)

print(f"Downloading: {model_zip}")
files.download(model_zip)